# Estimate Telework Arrangement Model

This notebook illustrates how to re-estimate a single model component for ActivitySim.  This process 
includes running ActivitySim in estimation mode to read household travel survey files and write out
the estimation data bundles used in this notebook.  To review how to do so, please visit the other
notebooks in this directory.

## Load libraries

In [14]:
import os
import larch
import pandas as pd
import activitysim

from activitysim.estimation.larch import P, component_model, update_coefficients

## Load estimation data bundle and prepare model for estimation

In [ ]:
modelname = "telework_arrangement"

from activitysim.estimation.larch import component_model

model, data = component_model(
    modelname,
    edb_directory=f"output-est-mode/estimation_data_bundle/{modelname}/",
    return_data=True,
)

## Review data loaded from the EDB

The next step is to read the EDB, including the coefficients, model settings, utilities specification, and chooser and alternative data.

### Coefficients

In [15]:
data.coefficients

,value,constrain
coefficient_name,,
coef_unavailable,-999.0,F
coef_acs_no_in_home_work_activity,-0.5,F


### Utility Specification

In [16]:
data.spec

,Label,Description,Expression,has_in_home_work_activity,no_in_home_work_activity
0,util_acs,alternative specific constant,@1,coef_acs_has_in_home_work_activity,NaN
1,util_part_time_worker,partime worker,ptype==2,coef_parttime_has_in_home,NaN
2,util_univ_student,university student,ptype==3,coef_univ_student_has_in_home,NaN
3,util_driving_age_student,driving age student,ptype==6,-999,NaN
4,util_has_preschool_at_home,has preschool children at home on the day,has_preschool_kid_at_home,coef_has_preschool_at_home_has_in_home,NaN
5,util_income_100_150,income 100-150k,(income_in_thousands>=100)&(income_in_thousand...,coef_income_100_150,NaN
6,util_income_150p,income 150k plus,income_in_thousands>=150,coef_income_150p,NaN
7,util_age_above_35,age above 35,age>=35,coef_age_above_35,NaN
8,util_workplace_mc_logsum,workplace mode choice logsum,"@np.where(df.workplace_zone_id>-1, df.workplac...",coef_workplace_mc_logsum,NaN
9,util_dummy_missing_workplace,util_dummy_missing_workplace,@np.where((df.workplace_zone_id<=-1)&(~df.work...,coef_dummy_missing_workplace,NaN


### Chooser Data

In [17]:
data.chooser_data

,person_id,model_choice,override_choice,util_acs,util_cdap_activity_mandatory,PNUM,age,sex,pemploy,pstudent,...,workplace_location_accessibility_0,workplace_location_accessibility_1,workplace_location_accessibility_2,othdiscr_accessibility_0,othdiscr_accessibility_1,othdiscr_accessibility_2,shopping_accessibility_0,shopping_accessibility_1,shopping_accessibility_2,override_choice_code
household_id,,,,,,,,,,,,,,,,,,,,,
43394,85660,False,True,1.0,0.0,1,27,1,1,3,...,12.439093,11.081183,11.010344,15.741796,14.071562,14.179500,12.927407,10.816278,11.013279,1
43394,85661,False,False,1.0,0.0,2,30,2,1,3,...,12.439093,11.081183,11.010344,15.741796,14.071562,14.179500,12.927407,10.816278,11.013279,2
43395,85662,True,False,1.0,0.0,1,43,2,1,3,...,11.285444,9.907935,9.963907,14.577302,12.350354,12.300278,11.764672,9.922871,10.299295,2
43396,85663,False,False,1.0,0.0,1,36,1,1,3,...,12.439265,10.933706,11.060478,16.539862,13.742941,15.129073,14.059675,11.687374,11.321886,2
43397,85664,True,False,1.0,0.0,1,41,2,1,2,...,11.294956,10.141743,10.397268,15.223566,14.333434,14.479188,13.017708,12.144687,11.655321,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49758,96877,False,True,1.0,0.0,2,54,1,1,3,...,11.169396,10.139722,9.993223,15.840142,13.464781,14.716995,12.269266,10.584173,10.758504,1
49759,96878,True,False,1.0,0.0,1,34,2,1,3,...,11.294216,10.716470,10.652009,15.718282,13.487907,13.567234,11.412606,9.752651,9.546404,2
49760,96879,False,False,1.0,0.0,1,55,2,2,3,...,11.516290,10.489083,10.382643,15.644795,14.272405,13.916283,12.885912,10.850070,10.857627,2


## Estimate

With the model setup for estimation, the next step is to estimate the model coefficients.  Make sure to use a sufficiently large enough household sample and set of zones to avoid an over-specified model, which does not have a numerically stable likelihood maximizing solution.  Larch has a built-in estimation methods including BHHH, and also offers access to more advanced general purpose non-linear optimizers in the `scipy` package, including SLSQP, which allows for bounds and constraints on parameters.  BHHH is the default and typically runs faster, but does not follow constraints on parameters.

In [18]:
model.estimate(method='SLSQP',maxiter=500)

,value,best,initvalue,minimum,maximum,nullvalue,holdfast
param_name,,,,,,,
-999,-999.000000,-999.000000,-999.0,-999.0,-999.0,0.0,1
coef_acs_has_in_home_work_activity,-1.387227,-1.387227,0.0,-50.0,50.0,0.0,0
coef_age_above_35,-0.206270,-0.206270,0.0,-50.0,50.0,0.0,0
coef_cdap_N_non_mandatory_accessibility,0.058416,0.058416,0.0,-50.0,50.0,0.0,0
coef_dummy_missing_workplace,0.433208,0.433208,0.0,-50.0,50.0,0.0,0
coef_has_preschool_at_home_has_in_home,0.341438,0.341438,0.0,-50.0,50.0,0.0,0
coef_income_100_150,0.128324,0.128324,0.0,-50.0,50.0,0.0,0
coef_income_150p,0.222101,0.222101,0.0,-50.0,50.0,0.0,0
coef_industry_accomodation,-0.797864,-0.797864,0.0,-50.0,50.0,0.0,0


┣                x: -999                                      -999.000000
┃                   coef_acs_has_in_home_work_activity          -1.387227
┃                   coef_age_above_35                           -0.206270
┃                   coef_cdap_N_non_mandatory_accessibility      0.058416
┃                   coef_dummy_missing_workplace                 0.433208
┃                   coef_has_preschool_at_home_has_in_home       0.341438
┃                   coef_income_100_150                          0.128324
┃                   coef_income_150p                             0.222101
┃                   coef_industry_accomodation                  -0.797864
┃                   coef_industry_construction                  -0.850865
┃                   coef_industry_education                     -0.541068
┃                   coef_industry_entertainment                 -1.090914
┃                   coef_industry_government                    -0.516547
┃                   coef_industry_healthcare                    -0.466444
┃                   coef_industry_manufacturing                 -0.415130
┃                   coef_industry_mgmt_srv                      -0.593968
┃                   coef_industry_military                      -0.644711
┃                   coef_industry_other                         -0.986052
┃                   coef_industry_retail                        -0.746360
┃                   coef_intrazonal                              0.585948
┃                   coef_parttime_has_in_home                   -0.630553
┃                   coef_telecommute_1_day_cdap_H                1.434154
┃                   coef_telecommute_1_day_cdap_N                0.397225
┃                   coef_telecommute_2_3_4p_days_cdap_N          1.814109
┃                   coef_telecommute_2_3_days_cdap_H             2.265012
┃                   coef_telecommute_4p_days_cdap_H              2.538976
┃                   coef_univ_student_has_in_home               -0.736627
┃                   coef_work_from_home_cdap_H                   2.807767
┃                   coef_work_from_home_cdap_N                   2.585807
┃                   coef_workplace_mc_logsum                    -0.034937
┃                   dtype: float64
┣          logloss: np.float64(0.481152516440139)
┣        d_logloss: -999                                       0.000000
┃                   coef_acs_has_in_home_work_activity         0.000024
┃                   coef_age_above_35                         -0.000163
┃                   coef_cdap_N_non_mandatory_accessibility    0.000003
┃                   coef_dummy_missing_workplace               0.000035
┃                   coef_has_preschool_at_home_has_in_home     0.000008
┃                   coef_income_100_150                        0.000121
┃                   coef_income_150p                           0.000115
┃                   coef_industry_accomodation                -0.000019
┃                   coef_industry_construction                -0.000047
┃                   coef_industry_education                   -0.000018
┃                   coef_industry_entertainment               -0.000025
┃                   coef_industry_government                  -0.000005
┃                   coef_industry_healthcare                   0.000058
┃                   coef_industry_manufacturing                0.000071
┃                   coef_industry_mgmt_srv                     0.000022
┃                   coef_industry_military                     0.000016
┃                   coef_industry_other                        0.000159
┃                   coef_industry_retail                      -0.000057
┃                   coef_intrazonal                           -0.000003
┃                   coef_parttime_has_in_home                  0.000224
┃                   coef_telecommute_1_day_cdap_H              0.000068
┃                   coef_telecommute_1_day_cdap_N             -0.000065
┃                   coef_telecommute_2_

In [19]:
model.estimation_statistics()

<xmle.Elem 'div' with 1 children>

### Estimated coefficients

In [20]:
model.parameter_summary()

,Value,Std Err,t Stat,Signif,Null Value,Constrained
Parameter,,,,,,
-999,-999.,0.00,NA,,0.00,fixed value
coef_acs_has_in_home_work_activity,-1.39,0.139,-9.95,***,0.00,
coef_age_above_35,-0.206,0.0714,-2.89,**,0.00,
coef_cdap_N_non_mandatory_accessibility,0.0584,0.00723,8.08,***,0.00,
coef_dummy_missing_workplace,0.433,0.142,3.05,**,0.00,
coef_has_preschool_at_home_has_in_home,0.341,0.180,1.90,,0.00,
coef_income_100_150,0.128,0.0837,1.53,,0.00,
coef_income_150p,0.222,0.0785,2.83,**,0.00,
coef_industry_accomodation,-0.798,0.211,-3.78,***,0.00,


## Output Estimation Results

In [ ]:
from activitysim.estimation.larch import update_coefficients

result_dir = data.edb_directory / "estimated"
update_coefficients(
    model,
    data,
    result_dir,
    output_file=f"{modelname}_coefficients_revised.csv",
)

### Write the model estimation report, including coefficient t-statistic and log likelihood

In [ ]:
model.to_xlsx(
    result_dir/f"telework_arrangement_model_estimation.xlsx", 
    data_statistics=True,
)

## Next Steps

The final step is to either manually or automatically copy the `*_coefficients_revised.csv` file to the configs folder, rename it to `*_coefficients.csv`, and run ActivitySim in simulation mode.

In [ ]:
pd.read_csv(result_dir / f"{modelname}_coefficients_revised.csv")